# 📈 All-in-One Gradient Descent Regressor

A custom linear regression implementation from scratch using NumPy that supports **Batch**, **Stochastic (SGD)**, and **Mini-Batch** Gradient Descent optimization strategies.

---

## 🎯 Project Goal
Build a flexible `GradientDescentRegressor` class with a scikit-learn–inspired API (`fit`, `predict`) that allows toggling between optimization methods while configuring key hyperparameters:
* **`method`**: `'batch'`, `'stochastic'`, or `'mini_batch'`
* **`lr`**: Learning rate (α)
* **`epochs`**: Number of full passes over the training dataset
* **`batch_size`**: Number of samples per gradient update (used when `method='mini_batch'`)

---

## 🧮 Mathematical Formulation

The model predicts outputs using the linear hypothesis:
\[\hat{y} = Xw + b\]

It minimizes the **Mean Squared Error (MSE)** loss function:
\[J(w, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2\]

At each step, parameters are updated using the gradients:
\[w := w - \alpha \frac{\partial J}{\partial w}, \quad b := b - \alpha \frac{\partial J}{\partial b}\]

| Variant | Samples per Update (k) | Update Frequency per Epoch | Key Trade-off |
| :--- | :---: | :---: | :--- |
| **Batch GD** | m (Full dataset) | 1 | Smooth, stable convergence; slow on large datasets |
| **Stochastic GD** | 1 | m | Fast updates, escapes local noise; high variance/oscillation |
| **Mini-Batch GD** | `batch_size` (1 < k < m) | \(\lceil m / k \rceil\) | Balances vectorization speed and gradient stability |

---

## ✅ Pass Conditions & Verification

To validate the implementation, run the regressor on synthetic data generated via `sklearn.datasets.make_regression()` and verify the following:

1. **Convergence Visualization (MSE vs. Epochs):**
   * Plot the training MSE curve across epochs for all three variants (`batch`, `stochastic`, `mini_batch`) on the same figure.
   * Confirm that Batch GD produces a smooth downward curve, while Stochastic and Mini-Batch GD converge faster per epoch with minor oscillations.

2. **Weight Equivalence Check:**
   * Compare your learned weights (`model.coef_`) and intercept (`model.intercept_`) against scikit-learn's closed-form Ordinary Least Squares solution (`sklearn.linear_model.LinearRegression`).
   * Verify close numerical agreement using `np.allclose()`:
     ```python
     assert np.allclose(custom_model.coef_, sklearn_model.coef_, atol=1e-2)
     assert np.isclose(custom_model.intercept_, sklearn_model.intercept_, atol=1e-2)
     ```

---

## 🚀 Example Usage

```python
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# 1. Generate synthetic dataset
X, y = make_regression(n_samples=500, n_features=5, noise=10.0, random_state=42)

# 2. Train custom regressor
gd_model = GradientDescentRegressor(method='mini_batch', lr=0.05, epochs=100, batch_size=32)
gd_model.fit(X, y)

# 3. Verify against Scikit-Learn
sk_model = LinearRegression().fit(X, y)

print("Custom Weights: ", np.round(gd_model.coef_, 4))
print("Sklearn Weights:", np.round(sk_model.coef_, 4))
```

---

## 🛠️ Tech Stack
* **Python 3.x**
* **NumPy** (Vectorized matrix operations)
* **Scikit-Learn** (Dataset generation & baseline verification)
* **Matplotlib** (Loss curve plotting)


In [126]:
import numpy as np 
import matplotlib.pyplot as plt 
from sklearn.datasets import make_regression
import random

In [127]:
class GradientDescentRegressor():

    ''' # method --> Select a method for Gradient Descent 
            1. 'batch' --> Will apply Batch Gradient Descent (Default)
            2. 'sgd' --> Will apply Stochastic Gradient Descent
            3. 'mini_batch' --> Will apply Mini Batch Gradient Descent

            # epoch --> Set the number of iteration for the Gradient Descent 
            The default value is 1000 iterations

            # learning_rate --> Set the learning rate
            To control how big each parameter update step is as the model moves toward the lowest loss
            default is 0.01

            # batch_size --> if method is 'mini_batch'
            Set the number of batches for update per epoch
            default is 35
            '''

    def __init__(self,method='batch',epoch=1000,learning_rate=0.01,batch_size=35):
        
        self.method = method 
        self.epoch = epoch
        self.lr = learning_rate
        self.batch_size = batch_size
        self.coef_ = None
        self.intercept_ = None

    def fit(self,X,Y):

        valid_methods = ['batch','sgd','mini_batch']
        
        if self.method not in valid_methods:
            raise ValueError(f"Invalid method: {self.method}. Expected one of {', '.join(valid_methods)}")

        # applicable for pandas dataframes
        X = np.asarray(X)
        Y = np.asarray(Y)

        
        if self.method == 'batch':

            self.coef_ = np.ones(X.shape[1])
            self.intercept_ = 0
            
            for i in range(self.epoch):
                Y_hat = np.dot(X,self.coef_) + self.intercept_
    
                intercept_der = -2*np.mean(Y-Y_hat)
                self.intercept_ = self.intercept_ - self.lr*intercept_der
    
                coef_der = (-2/X.shape[0])*np.dot((Y-Y_hat),X)
                self.coef_ = self.coef_ - self.lr*coef_der

        elif self.method == 'sgd':

            self.coef_ = np.ones(X.shape[1])
            self.intercept_ = 0

            for i in range(self.epoch):

                for j in range(X.shape[0]):

                    idx = np.random.randint(0,X.shape[0])

                    Y_hat = np.dot(X[idx],self.coef_) + self.intercept_
    
                    intercept_der = -2*np.mean(Y[idx]-Y_hat)
                    self.intercept_ = self.intercept_ - self.lr*intercept_der
        
                    coef_der = -2*np.dot((Y[idx]-Y_hat),X[idx])
                    self.coef_ = self.coef_ - self.lr*coef_der

        elif self.method == 'mini_batch':

            self.coef_ = np.ones(X.shape[1])
            self.intercept_ = 0

            for i in range(self.epoch):

                for j in range(X.shape[0] // self.batch_size):

                    idx = random.sample(range(X.shape[0]),self.batch_size)
                    Y_hat = np.dot(X[idx],self.coef_) + self.intercept_
    
                    intercept_der = -2*np.mean(Y[idx]-Y_hat)
                    self.intercept_ = self.intercept_ - self.lr*intercept_der
        
                    coef_der = (-2/X[idx].shape[0])*np.dot((Y[idx]-Y_hat),X[idx])
                    self.coef_ = self.coef_ - self.lr*coef_der
            
    def predict(self,x_test):

        if self.coef_ is None or self.intercept_ is None:
            raise ValueError('Cannot predict call fit first')

        x_test = np.asarray(x_test)
        
        return np.dot(x_test,self.coef_) + self.intercept_

In [128]:
X,Y = make_regression(n_samples=200,n_features=2,n_targets=1,n_informative=1,noise=20,random_state=2)

In [129]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [130]:
X_train,x_test,Y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [131]:
# applying sklearn linear regression
lr = LinearRegression()
lr.fit(X_train,Y_train)
print(f'R2 Score : {r2_score(y_test,lr.predict(x_test))}')
print(f'Coef : {lr.coef_}')
print(f'Intercept : {lr.intercept_}')

R2 Score : 0.9252480467214312
Coef : [ 3.18230413 70.15380906]
Intercept : -1.6845805618253822


In [132]:
# batch gradient descent
gd = GradientDescentRegressor()
gd.fit(X_train,Y_train)
y_pred = gd.predict(x_test)
print(f'R2 Score : {r2_score(y_test,y_pred)}')
print(f'Coef : {gd.coef_}')
print(f'Intercept : {gd.intercept_}')

R2 Score : 0.9252480466593423
Coef : [ 3.18230399 70.153809  ]
Intercept : -1.6845803819734044


In [133]:
# stochastic gradient descent
sgd = GradientDescentRegressor(method='sgd',epoch=300,learning_rate=0.01)
sgd.fit(X_train,Y_train)
y_pred = sgd.predict(x_test)
print(f'R2 Score : {r2_score(y_test,y_pred)}')
print(f'Coef : {sgd.coef_}')
print(f'Intercept : {sgd.intercept_}')

R2 Score : 0.924013772166944
Coef : [ 1.94073275 71.25568829]
Intercept : -3.8411635864625664


In [134]:
# mini batch gradient descent
mbgd = GradientDescentRegressor(method='mini_batch',epoch=1000,learning_rate=0.01)
mbgd.fit(X_train,Y_train)
y_pred = mbgd.predict(x_test)
print(f'R2 Score : {r2_score(y_test,y_pred)}')
print(f'Coef : {mbgd.coef_}')
print(f'Intercept : {mbgd.intercept_}')

R2 Score : 0.9254985011564211
Coef : [ 3.04281886 69.96099915]
Intercept : -1.9063982179505754
